In [ ]:
!pip install -e ..

In [ ]:
import numpy as np
%load_ext autoreload
%autoreload 0

Notebook by **Maxime Dion** <maxime.dion@usherbrooke.ca><br>
For the QSciTech-QuantumBC virtual workshop on gate-based quantum computing

# Important

When you modify and save a `*.py` file you need to re-import it so that your modifications can be taken into account when you re-execute a call. By adding the magic command `%autoreload` at the beginning of a cell, you make sure that the modifications you did to the `*.py` files are taken into account when you re-run a cell and that you can see the effect.

If you encounter weird behavior, restart the kernel and try again.

# Tutorial 1 (Mapping)

In this tutorial you will complete the implementation of the classes `PauliString` and `Operator` which will be useful to build a qubit representation of the Hamiltonian of a small molecule. You will then implement the Jordan Wigner mapping.

By completing all sections of this notebook you will be able to :
- Create `PauliString` instances.
- Multiply `PauliString` together.
- Translate a `PauliString` into a unitary matrix of size `(2**n)**2`.
- Create `Operator` instances.
- Multiply and add `Operator` together.
- Combine repeated `PauliString`s in `Operator`.
- Translate a `Operator` into a matrix of size `(2**n)**2`.
- Use the Jordan-Wigner mapping to translate a Fermionic Hamiltonian into a Qubit Hamiltonian as an instance of an `Operator`.

The solution we suggest here is NOT mandatory. If you find ways to make it better and more efficient, go on and impress us! 

**Remark on qiskit**

We use `qiskit` in this workshop. The tools you'll be building here are already available in `qiskit`. For instance, the `PauliString` and `Operator` classes here have similar purpuse to `Pauli` and `SparsePauliOp` in `qiskit`. Nevertheless, we strongly encourage you to complete the current implementation because we think it as a valuable learning experience.

At the end of the workshop, we encourage you to use `qiksit` to complete the same project. It should give you the same results, but now you'll understand what is going on under the hood.

# PauliString

The `PauliString` class is partially implemented in the file `Pauli.py`.

In [ ]:
from quantum_chemistry.pauli import PauliString

## Creation

This object's attributes are 2 arrays of booleans `z_bits` and `x_bits`. You can easily create an instance and print the result. The `__str__` method is already implemented so you can use `print()` on any instance of `PauliString`. The boolean arrays in input are in the `0123` order and the label string is in the reversed `q3q2q1q0` order. Here we initialize the Pauli string `YXZI`.

In [ ]:
z_bits = np.array([0,1,0,1], dtype = bool)
x_bits = np.array([0,0,1,1], dtype = bool)
pauli_string = PauliString(z_bits, x_bits)
print(pauli_string)

### Creation exercice

Create the `ZZXY` Pauli string. Remember that the arrays are in the 0...n order, but the string representation in the reverse order. The `print` should return `ZZXY`.

In [ ]:
z_bits = 
x_bits = 
print(PauliString(z_bits, x_bits))

### Creation from string

To create `PauliString` using alternative input data, you can implement `@classmethod`. 

Such a method called `from_str` is partially implemented in the `PauliString` class. Complete this implementation so that you can use it to create `PauliString` from a string such as `YXZI`.

In [ ]:
%autoreload
pauli_string = PauliString.from_str('YXZI')
print(pauli_string)

### Useful methods

In order to compare `PauliString`s together it's convenient to represent it as `zx_bits` which is an array twice as long that combines `z_bits` and `x_bits`. Implement the `to_zx_bits()` method. Why not do the `to_xz_bits()` while your at it!

You should get :

<code>
[False  True False  True False False  True  True]<br>[False False  True  True False  True False  True]
</code>

In [ ]:
%autoreload
pauli_string = PauliString.from_str('YXZI')
zx_bits = pauli_string.to_zx_bits()
print(zx_bits)
xz_bits = pauli_string.to_xz_bits()
print(xz_bits)

It's also useful to know where are the $\hat{I}$ in a `PauliString`. Implement the method that does this. You should get 

<code>[ True False False False]</code>

In [ ]:
%autoreload
pauli_string = PauliString.from_str('YXZI')
ids = pauli_string.ids()
print(ids)

## Multiplication with another PauliString

Multiplying `PauliString`s is essential to be able to translate Fermionic Hamiltonians into a qubit Hamiltonian. 

Before you implement the method that will allow you to do this, you should experiment a bit with how boolean arrays behave. Take a look at methods like `np.dot()`, `np.logical_and()`, `np.logical_or()` and `np.logical_xor()`. In particular, notice that the addition `+` on booleans is not a (mod 2) addition (it's a `logical_or`) and the `np.sum()` method on a boolean array counts the number of 1 and returns an `int`.

In [ ]:
bits_1 = np.array([0,1,0,1], dtype = bool)
bits_2 = np.array([0,1,1,1], dtype = bool)
print(bits_1 + bits_2)
print(np.sum(bits_1))
# Experiment

With these considerations, implement the `mul_pauli_string()` method in order to replicate the product

\begin{align}
\hat{I}\hat{Y}\hat{Z}\hat{Z} \times \hat{I}\hat{I}\hat{X}\hat{Z}  = i \hat{I}\hat{Y}\hat{Y}\hat{I}.
\end{align}

The product return a `PauliString` and a phase (`complex`). The method `__mul__` is already implemeted to call `mul_pauli_string()` so you can use `*` to do the product.

In [ ]:
%autoreload
pauli_string_1 = PauliString.from_str('IYZZ')
pauli_string_2 = PauliString.from_str('IIXZ')
new_pauli_string, phase = pauli_string_1 * pauli_string_2
print(new_pauli_string, phase)

Check your solution on many pairs of Pauli strings such as

\begin{align}
\hat{Z}\hat{Z}\hat{Z}\hat{Z} \times \hat{X}\hat{X}\hat{X}\hat{I}  = -i \hat{Y}\hat{Y}\hat{Y}\hat{Z}.
\end{align}

In [ ]:
%autoreload
pauli_string_1 = PauliString.from_str('ZZZZ')
pauli_string_2 = PauliString.from_str('XXXI')
new_pauli_string, phase = pauli_string_1 * pauli_string_2
print(new_pauli_string, phase)

## Matrix representation
The matrix reprensetation of `PauliString` will only be used to compute the exact solution of the Hamiltonian. It will not be used for quantum computing, but it's a nice way to validate your results.

Any `PauliString` can be converted into a matrix. This is useful to find the exact solution for small systems. To combine the space of two qubits we use the [Kronecker product](https://en.wikipedia.org/wiki/Kronecker_product) ($\otimes$). For example, the `ZX` Pauli string can be represented as the following matrix

\begin{align}
    \hat{Z}_1\hat{X}_0 = \hat{Z}_1\otimes\hat{X}_0 &= \begin{pmatrix} 1 \times \hat{X}_0 & 0 \\ 0 & -1 \times \hat{X}_0 \end{pmatrix} \\
    &= \begin{pmatrix} 0 & 1 & 0 & 0 \\ 1 & 0 & 0 & 0 \\ 0 & 0 & 0 & -1 \\ 0 & 0 & -1 & 0 \end{pmatrix}
\end{align}

which is expressed in the basis

\begin{align}
    |00\rangle, |01\rangle, |10\rangle, |11\rangle.
\end{align} 

Indeed we verify that

\begin{align}
    \hat{Z}_1\hat{X}_0 |10\rangle &= - |11\rangle \\
    \begin{pmatrix} 0 & 1 & 0 & 0 \\ 1 & 0 & 0 & 0 \\ 0 & 0 & 0 & -1 \\ 0 & 0 & -1 & 0 \end{pmatrix}\begin{pmatrix} 0 \\ 0 \\ 1 \\ 0 \end{pmatrix} &= -\begin{pmatrix} 0 \\ 0 \\ 0 \\ 1 \end{pmatrix}
\end{align}

The `np.kron()` method is a straight foward way to acheive this. The previous 2 `PauliString`s can be turned into a $4\times 4$ matrix like this.

In [ ]:
z_matrix = np.array([[1, 0], [0, -1]], dtype = int)
x_matrix = np.array([[0, 1], [1, 0]], dtype = int)
print(z_matrix)
print(x_matrix)
zx_matrix = np.kron(z_matrix, x_matrix)
print(zx_matrix)

Implement the `to_matrix()` method for any Pauli string and try it to find the matrix form of `ZX`.

In [ ]:
%autoreload
pauli_string = PauliString.from_str('ZX')
print(pauli_string)
matrix = pauli_string.to_matrix()
print(matrix)

# Operator
The `Operator` class is partially implemented in the file `Pauli.py`.

In [ ]:
from quantum_chemistry.pauli import Operator

## Creation

To build a `Operator` you only need to provide an `numpy.array` of coefficients (`complex`) and a `numpy.array` of `PauliString`. If they are not arrays, they will be converted. Here again the `__str__()` method is already implemented.

In [ ]:
coefs = np.array([0.5, 0.5], dtype = complex)
pauli_string_1 = PauliString.from_str('IIXZ')
pauli_string_2 = PauliString.from_str('IYZZ')
pauli_strings = np.array([pauli_string_1, pauli_string_2], dtype = PauliString)
operator = Operator(coefs,pauli_strings)
print(operator)

### Multiplication of a PauliString by a coefficient

Multiplying a `PauliString` by a number is a useful way to create a `Operator` with only 1 `PauliString`. Implement the method `mul_coef` in the `PauliString` class so that you can easily create a `Operator`.

In [ ]:
%autoreload
operator_pauli = 1 * PauliString.from_str('IIXZ')
print(operator_pauli)

### Adding of Operators
The sum of two `Operator`s is just the union of these two ensembles. Implement `add_operator()` and test your solution here. The `__add__()` method is already implemented to call `add_operator()` so you can use the `+` operator.

In [ ]:
%autoreload
operator = 0.5*pauli_string_1 + 0.5*pauli_string_2
print(operator)

### Product of Operators

The product of two `Operator`s can be computed using the distributive property of any sum. While you implement `mul_operator()` make sure you take into account the phase coming from the product of two `PauliString`s. You can test your code on the following cell.

In [ ]:
%autoreload
operator_1 = 1*PauliString.from_str('IIXZ')
operator_2 = 1*PauliString.from_str('IYZZ')
new_operator = operator_1 * operator_2
print(new_operator)

You should get:

<code>
(0.00-1.00j)*IYYI
</code>

With addition and multiplication, `Operator`s are much more convenient to work with than `PauliString`s because they carry the possible phase from the product.

## Accessing subset of an Operator
A `__getitem__()` method is already implemented to access subset of the `Operator`. You can use indices and slices, like a `list` or an `np.array`.

In [ ]:
operator = 1*PauliString.from_str('IIIZ') + 1*PauliString.from_str('IIZI') + 1*PauliString.from_str('IZII') + 1*PauliString.from_str('ZIII')
print(operator[0])
print(operator[1:3])
print(operator[-1])

## Combinaison and threshold
When a `PauliString` is present many times in a `Operator`, it is convenient to be able to remove extra occurences by combining the respective coefficients. Let's take the example from the presentation.

In [ ]:
operator_1 = 1*PauliString.from_str('IIIZ') - 0.5*PauliString.from_str('IIZZ') 
operator_2 = 1*PauliString.from_str('ZZZI') + 0.5*PauliString.from_str('ZZII') 
operator_3 = operator_1 * operator_2
print(operator_3)

We see that `ZZZZ` occurs 2 times and `ZZIZ` occurs 2 times as well.

Implement the `combine()` method to reduce the `Operator` to 2 `PauliString`s. There are many ways to do that. Suggestion, convert with `to_zx_bits()` and use the `np.unique()` method. DO NOT remove `PauliString`s with `0` coef yet.

In [ ]:
%autoreload
operator_combined = operator_3.combine()
print(operator_combined)

You should get:

<code>
(0.75+0.00j)*ZZZZ + (0.00+0.00j)*ZZIZ
</code>

Implement the `apply_threshold()` method to get rid of any Pauli string with a coefficient smaller than the `threshold`.

In [ ]:
%autoreload
operator = operator_combined.apply_threshold()
print(operator)

You should get:

<code>
(0.75+0.00j)*ZZZZ
</code>

## Matrix representation

The `Operator` can also be represented as a matrix. This matrix is just the linear combinaison of the matrices representing each Pauli string. Implement the `to_matrix()` method.

In [ ]:
%autoreload
small_operator = 1*PauliString.from_str('ZZ') + 2*PauliString.from_str('XX')
matrix = small_operator.to_matrix()
print(matrix)

You should get :

<code>
[[ 1.+0.j  0.+0.j  0.+0.j  2.+0.j]<br> [ 0.+0.j -1.+0.j  2.+0.j  0.+0.j]<br> [ 0.+0.j  2.+0.j -1.+0.j  0.+0.j]<br> [ 2.+0.j  0.+0.j  0.+0.j  1.+0.j]] 
</code>

# Molecular Hamiltonian

Before constructing the qubit representation of the molecular Hamiltonian we need the Hamiltonian in its fermionic representation. Therefore, we first need to get the integral tensors. You can use either one of the two solutions presented here depending on how succesful you were with the installation of the now infamous `pyscf` module.

- We will first show how to load the integrals from the precomputed files located in the `h2_data` directory.
- Then we will give you instructions to generate these files (and more) using `pyscf`. (optionnal)


### Loading the integrals from files

The integrals are precomputed for a given set of interatomic distances. To load the data for one distance use the following code. The first ouput is the distance. The next two are the 2d and 4d tensors for the one body and two body integrals. The last is the nuclear repulsion energy which will contribute to the total energy.

In [ ]:
from quantum_chemistry.molecule.h2_molecule import load_h2_spin_orbital_integral

distance, one_body, two_body, nuc_eneg = load_h2_spin_orbital_integral("../h2_data","h2_mo_integrals_d_0750.npz")

You can also load them all at one. It will return you a list of distances and a list of tuples structured such this way `(one_body, two_body, nuc_eneg)`. 

In [ ]:
from quantum_chemistry.molecule.h2_molecule import load_h2_spin_orbital_integrals

distances, molecule_datas = load_h2_spin_orbital_integrals("../h2_data")

### Generating integrals

If you were successful in installing `pyscf` you can generate the integrals yourself. You can use the following code to generate data files for given distances and save them to a given location. You'll be able to load them later.

In [ ]:
from quantum_chemistry.molecule.h2_molecule import generate_and_save_h2_spin_orbital_integrals

distances = distances = np.round(np.arange(0.3, 2, 0.05), 2)

generate_and_save_h2_spin_orbital_integrals(distances, "../h2_data")

Alternatively, you can also generate the integrals on the fly given a single interatomic distance.

In [ ]:
from quantum_chemistry.molecule.h2_molecule import get_h2_spin_orbital_integrals

one_body, two_body, nuclear_repulsion_energy = get_h2_spin_orbital_integrals(0.735)

You could also use this code to generate integrals for other molecules.

# Mapping
You are now in good position to implement your first mapping. The functions to perform the Jordan-Wigner Mapping are partially implemented in the file `Mapping.py`.

## Jordan-Wigner creation/annihilation operators

The first task of the mapping is to translate creation and annihilation fermionic operators into `Operator`. You now need to implement the `creation_annihilation_operators_with_jordan_wigner()` function. It should return 2 lists of `Operator`s, one `list` for creation operators and one `list` for annihilation operators. You can make use of the addition and multipliation method you implemented earlier.

Refer to the presentation for the general structure of the Jordan-Wigner mapping. Make sure your method works for different numbers of qubits.

In [ ]:
%autoreload
from quantum_chemistry.mapping import creation_annihilation_operators_with_jordan_wigner

creation_operators, annihilation_operators = creation_annihilation_operators_with_jordan_wigner(4)
print(len(creation_operators), 'creation operators')
print('Creation operators')
for ap in creation_operators:
    print(ap)
print()
print(len(annihilation_operators), 'annihilation operators')
print('Annihilation operators')
for am in annihilation_operators:
    print(am)

For the creation operators you should get.

<code>
4 creation operators<br>
Creation operators<br>
(0.50+0.00j)*IIIX + (-0.00-0.50j)*IIIY<br>
(0.50+0.00j)*IIXZ + (-0.00-0.50j)*IIYZ<br>
(0.50+0.00j)*IXZZ + (-0.00-0.50j)*IYZZ<br>
(0.50+0.00j)*XZZZ + (-0.00-0.50j)*YZZZ<br>
</code>

For the annihilation just reverse the sign of the imaginary part.

## Building the Qubit Hamiltonian

The construct the qubit Hamiltonian, you only need to use the mapped creation and annihilation operators and compose them following the Hamiltonien formulation.

### One body term

Let's start with the one_body part. The one body Hamiltonian is of the form 

\begin{align*}
    \hat{H}_1 = \sum_{i,j} h_{ij} \hat{a}_i^\dagger \hat{a}_j
\end{align*} 

You should now be able to implement the `build_one_body_qubit_hamiltonian` function in the `mapping.py` file. 

Each fermionic operator is made of 2 Pauli strings. So each term $i,j$ creates 4 Pauli strings. There is 16 terms in $h_{ij}$ so there is 64 Pauli strings (if we do not consider combinaison and applying threshold yet). Your implementation should now return a `Operator` of length 64.

In [ ]:
%autoreload
from quantum_chemistry.mapping import build_one_body_qubit_hamiltonian, creation_annihilation_operators_with_jordan_wigner
from quantum_chemistry.molecule.h2_molecule import load_h2_spin_orbital_integral

distance, one_body, two_body, nuc_eneg = load_h2_spin_orbital_integral("../h2_data","h2_mo_integrals_d_0750.npz")

creation_operators, annihilation_operators = creation_annihilation_operators_with_jordan_wigner(4)

one_body_operator = build_one_body_qubit_hamiltonian(one_body, creation_operators, annihilation_operators)


We see many Pauli strings with 0 coefficient as well as many repeated strings. We can now exploit `combine()` and `apply_threshold()`. Since there is many 0 terms already we do `apply_threshold()` first, then `combine()` and `apply_threshold()` again if there was any cancellations. We can finish with `sort()` for neat presentation.

In [ ]:
one_body_operator = one_body_operator.apply_threshold().combine().apply_threshold().sort()
print(one_body_operator)

You should get :

<code>
(-1.73+0.00j)*IIII + (0.62+0.00j)*IIIZ + (0.62+0.00j)*IZII + (0.24+0.00j)*IIZI + (0.24+0.00j)*ZIII
</code>

### Two body term
The two body Hamiltonian is of the following form :

\begin{align*}
    \hat{H}_2 = \frac{1}{2}\sum_{i,j} h_{ijkl} \hat{a}_i^\dagger\hat{a}_j^\dagger \hat{a}_k\hat{a}_l
\end{align*} 

You can now implement the `build_two_body_qubit_hamiltonian` function in the `mapping.py` file. 
Counting all Pauli strings you should produce 4096 Pauli strings.

In [ ]:
%autoreload
from quantum_chemistry.mapping import build_two_body_qubit_hamiltonian, creation_annihilation_operators_with_jordan_wigner
from quantum_chemistry.molecule.h2_molecule import load_h2_spin_orbital_integral

distance, one_body, two_body, nuc_eneg = load_h2_spin_orbital_integral("../h2_data","h2_mo_integrals_d_0750.npz")
creation_operators, annihilation_operators = creation_annihilation_operators_with_jordan_wigner(4)

two_operator = build_two_body_qubit_hamiltonian(two_body, creation_operators, annihilation_operators)

Applying the `combine` and `apply_threshold` this reduce to 15 Pauli strings!

In [ ]:
two_operator = two_operator.apply_threshold().combine().apply_threshold().sort()
print(two_operator)

You should get :

<code>
(1.83+0.00j)*IIII<br>
(-0.92+0.00j)*ZIII<br>
(-0.92+0.00j)*IIZI<br>
(-0.91+0.00j)*IIIZ<br>
(-0.91+0.00j)*IZII<br>
(0.35+0.00j)*ZIZI<br>
(0.34+0.00j)*IZIZ<br>
(0.33+0.00j)*ZIIZ<br>
(0.33+0.00j)*IZZI<br>
(0.24+0.00j)*IIZZ<br>
(0.24+0.00j)*ZZII<br>
(0.09+0.00j)*YYYY<br>
(0.09+0.00j)*XXYY<br>
(0.09+0.00j)*YYXX<br>
(0.09+0.00j)*XXXX<br>
</code>

### Molecular Hamiltonian

The molecular Hamiltonian is just the sum of the one and two body terms. Implement th function `build_qubit_hamiltonian` in `mapping.py`. You should now be able to run this code.

In [ ]:
%autoreload
from quantum_chemistry.mapping import build_qubit_hamiltonian, creation_annihilation_operators_with_jordan_wigner
from quantum_chemistry.molecule.h2_molecule import load_h2_spin_orbital_integral

distance, one_body, two_body, nuc_eneg = load_h2_spin_orbital_integral("../h2_data","h2_mo_integrals_d_0750.npz")
creation_operators, annihilation_operators = creation_annihilation_operators_with_jordan_wigner(4)

qubit_hamiltonian = build_qubit_hamiltonian(one_body, two_body, creation_operators, annihilation_operators)

print(qubit_hamiltonian)

You should get :

<code>
(-0.82+0.00j)*IIII<br>
(-0.22+0.00j)*IIZI<br>
(-0.22+0.00j)*ZIII<br>
(0.17+0.00j)*ZIZI<br>
(0.17+0.00j)*IIIZ<br>
(0.17+0.00j)*IZII<br>
(0.17+0.00j)*IZIZ<br>
(0.17+0.00j)*ZIIZ<br>
(0.17+0.00j)*IZZI<br>
(0.12+0.00j)*IIZZ<br>
(0.12+0.00j)*ZZII<br>
(0.05+0.00j)*YYYY<br>
(0.05+0.00j)*XXYY<br>
(0.05+0.00j)*YYXX<br>
(0.05+0.00j)*XXXX<br>
</code>

## You have completed your first mapping of H2!

What now? The next step is to use this mapping to evaluate the Hamiltonian on a quantum computer. This is the topic of the next tutorial.


Notebook by **Maxime Dion** <maxime.dion@usherbrooke.ca><br>
For the QSciTech-QuantumBC virtual workshop on gate-based quantum computing